# Phase 3 · S2 — ALS ↔ chainRec head-to-head + cầu nối ID cho S3

**Mục tiêu**
1. **So ALS vs chainRec head-to-head** trên *cùng* `data_test`, *cùng* item-pool (1.57M), *cùng* mask, *cùng* `rank_eval` → AUC apples-to-apples.
2. Dựng **cầu nối ID** `int (UCSD interactions) ↔ string (reviews Phase 2)` để S3 gắn review-length (F3) vào chains.

**Quyết định thiết kế (quan trọng — ghi vào báo cáo):**
ALS Phase 2 huấn luyện trên *reviews* (ID string, split thời gian) và **không lưu factor**; chainRec
huấn luyện trên *interactions* (ID int). Hai universe khác nhau → bridge hai model đã-train-độc-lập
sẽ vướng leakage + overlap nhỏ. Nên ở đây ta **huấn luyện lại ALS NGAY trong index-space của chainRec**
(trên chính `data_train` recommend-edges). Khi đó user/item/test/pool/mask **giống hệt** chainRec,
chỉ khác **kiến trúc model** ⇒ phép so AUC đo đúng "monotonic-chain có hơn MF thuần trên cùng dữ liệu không".

**Nền tảng:** Kaggle GPU T4. Artifact load từ `/kaggle/working` hoặc HF `vngclinh/goodreads-preprocessed`.


## 0 · Setup

In [ ]:
import os, json, time, pickle, random
from pathlib import Path
from dataclasses import dataclass
from typing import Optional, Literal
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F

try:
    import implicit
except ImportError:
    os.system("pip install -q implicit"); import implicit
from scipy.sparse import csr_matrix

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "| implicit", implicit.__version__)

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass

## 1 · Config + load artifacts chainRec (local-or-HF)

In [ ]:
HF_REPO    = "vngclinh/goodreads-preprocessed"
PROC_LOCAL = Path("/kaggle/working/processed")
CKPT_LOCAL = Path("/kaggle/working/chainrec")
S2_LOCAL   = Path("/kaggle/working/s2"); S2_LOCAL.mkdir(parents=True, exist_ok=True)

EVAL_USERS  = 5000      # full-ranking eval trên 5000 test users (đủ ổn định, nhanh)
BATCH_USERS = 64
K_LIST      = (10, 20)

from huggingface_hub import hf_hub_download
def _hf(rel): return hf_hub_download(HF_REPO, rel, repo_type="dataset", token=HF_TOKEN)
def load_npy(n):
    p = PROC_LOCAL/n; return np.load(p if p.exists() else _hf(f"chainrec/processed/{n}"))
def load_pkl(n):
    p = PROC_LOCAL/n; return pickle.load(open(p if p.exists() else _hf(f"chainrec/processed/{n}"), "rb"))
def load_meta():
    p = PROC_LOCAL/"meta.json"
    return json.loads(Path(p if p.exists() else _hf("chainrec/processed/meta.json")).read_text())
def ckpt_path(s, tag=""):
    fn = f"chainrec_{s}{tag}.pt"; p = CKPT_LOCAL/fn
    return str(p) if p.exists() else _hf(f"chainrec/{fn}")

data_train    = load_npy("data_train.npy")
data_test     = load_npy("data_test.npy")
user_item_map = load_pkl("user_item_map.pkl")
meta          = load_meta()
N_ITEM, N_USER, N_STAGE = meta["n_item"], meta["n_user"], meta["n_stage"]
REC = N_STAGE - 1
print(f"n_user={N_USER:,}  n_item={N_ITEM:,}  n_stage={N_STAGE}  REC(stage recommend)={REC}")
print("train stage dist:", np.bincount(data_train[:,2], minlength=N_STAGE).tolist())
print("test  stage dist:", np.bincount(data_test[:,2],  minlength=N_STAGE).tolist())

## 2 · chainRec model (định nghĩa giống S0/S1a để load checkpoint)

In [ ]:
@dataclass
class ModelConfig:
    n_user: int; n_item: int
    n_stage: int = 4; embed_dim: int = 16
    beta: float = 1.0; learn_beta: bool = True
    l2: float = 0.01; lr: float = 0.001
    batch_size: int = 2048; n_neg: int = 1
    n_epochs: int = 30; patience: int = 5
    sampler: Literal["uniform","stagewise"] = "uniform"
    device: str = DEVICE

class ChainRecModel(nn.Module):
    def __init__(self, cfg):
        super().__init__(); self.cfg = cfg
        K, L = cfg.embed_dim, cfg.n_stage
        self.user_emb  = nn.Embedding(cfg.n_user, K)
        self.item_emb  = nn.Embedding(cfg.n_item, K)
        self.stage_emb = nn.Embedding(L, K)
        self.b0        = nn.Parameter(torch.zeros(1))
        self.b_user    = nn.Embedding(cfg.n_user, 1)
        self.b_item    = nn.Embedding(cfg.n_item, 1)
        lb = torch.log(torch.tensor(float(cfg.beta)))
        if cfg.learn_beta: self.log_beta = nn.Parameter(lb)
        else: self.register_buffer("log_beta", lb)
        for e in [self.user_emb, self.item_emb, self.stage_emb]: nn.init.xavier_uniform_(e.weight)
        for b in [self.b_user, self.b_item]: nn.init.zeros_(b.weight)
    @property
    def beta(self): return torch.clamp(self.log_beta.exp(), min=1.0)
    def _intention(self, u, i, l): return (self.stage_emb(l)*self.item_emb(i)*self.user_emb(u)).sum(-1)
    def _rect(self, d): b=self.beta; return F.softplus(b*d)/b
    def score(self, u, i, target_stage):
        B = u.shape[0]
        bias = self.b0 + self.b_user(u).squeeze(-1) + self.b_item(i).squeeze(-1)
        acc = torch.zeros(B, device=u.device)
        for lp in range(target_stage, self.cfg.n_stage):
            l_t = torch.full((B,), lp, dtype=torch.long, device=u.device)
            acc = acc + self._rect(self._intention(u, i, l_t))
        return bias + acc

def build_model(sampler="uniform"):
    cfg = ModelConfig(n_user=N_USER, n_item=N_ITEM, n_stage=N_STAGE, sampler=sampler)
    return ChainRecModel(cfg).to(DEVICE), cfg

## 3 · `rank_eval` model-agnostic + scorers (giống hệt S1a)

`rank_eval(score_fn, ...)` — full-ranking, mask seen items. AUC = metric so-với-paper.
3 scorer cùng một chữ ký `score_fn(u)->(B,n_item)`: chainRec / ALS / itemPop → cùng hàm eval.

In [ ]:
@torch.no_grad()
def make_chainrec_scorer(model, target_stage):
    model.eval()
    item_emb = model.item_emb.weight; b_item = model.b_item.weight.squeeze(-1)
    stage_w  = model.stage_emb.weight
    def score_fn(u):
        uvec = model.user_emb(u)
        bias = (model.b0 + model.b_user(u).squeeze(-1)).unsqueeze(1)
        acc  = torch.zeros(u.shape[0], item_emb.shape[0], device=u.device)
        for l in range(target_stage, model.cfg.n_stage):
            acc = acc + model._rect((uvec * stage_w[l].unsqueeze(0)) @ item_emb.t())
        return bias + b_item.unsqueeze(0) + acc
    return score_fn

@torch.no_grad()
def make_als_scorer(user_factors, item_factors, item_bias=None, device="cuda"):
    U  = torch.as_tensor(user_factors, dtype=torch.float32, device=device)
    V  = torch.as_tensor(item_factors, dtype=torch.float32, device=device)
    bi = None if item_bias is None else torch.as_tensor(item_bias, dtype=torch.float32, device=device)
    def score_fn(u):
        s = U[u] @ V.t()
        if bi is not None: s = s + bi.unsqueeze(0)
        return s
    return score_fn

@torch.no_grad()
def make_pop_scorer(item_pop, device="cuda"):
    p = torch.as_tensor(item_pop, dtype=torch.float32, device=device)
    def score_fn(u): return p.unsqueeze(0).expand(u.shape[0], -1)
    return score_fn

@torch.no_grad()
def rank_eval(score_fn, test_pairs, user_item_map, n_item, pos_stage=None,
              K_list=(10,20), batch_users=64, n_eval_users=None, device="cuda", seed=999):
    pairs = test_pairs if pos_stage is None else test_pairs[test_pairs[:,2] == pos_stage]
    if n_eval_users is not None and len(pairs) > n_eval_users:
        rng = np.random.default_rng(seed)
        pairs = pairs[rng.choice(len(pairs), size=n_eval_users, replace=False)]
    aucs=[]; hits={k:[] for k in K_list}; ndcg={k:[] for k in K_list}
    for st in range(0, len(pairs), batch_users):
        chunk = pairs[st:st+batch_users]
        u   = torch.tensor(chunk[:,0], dtype=torch.long, device=device)
        pos = torch.tensor(chunk[:,1], dtype=torch.long, device=device)
        scores = score_fn(u)
        B = u.shape[0]; ar = torch.arange(B, device=device)
        pos_score = scores[ar, pos].clone()
        seen_cnt = torch.zeros(B, device=device)
        for b in range(B):
            seen = user_item_map.get(int(u[b]), ())
            if seen:
                idx = torch.tensor(list(seen), dtype=torch.long, device=device)
                scores[b, idx] = float("-inf"); seen_cnt[b] = len(seen)
        scores[ar, pos] = pos_score
        rank = (scores > pos_score.unsqueeze(1)).sum(1).float()
        neg  = (n_item - seen_cnt).clamp(min=1)
        aucs.append((1.0 - rank/neg).cpu().numpy())
        rnp = rank.cpu().numpy()
        for k in K_list:
            hit = rnp < k
            hits[k].append(hit.astype(float))
            ndcg[k].append(np.where(hit, 1.0/np.log2(rnp+2), 0.0))
        del scores
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    res = {"AUC": float(np.concatenate(aucs).mean()), "n_eval": int(len(pairs))}
    for k in K_list:
        res[f"Recall@{k}"] = float(np.concatenate(hits[k]).mean())
        res[f"NDCG@{k}"]   = float(np.concatenate(ndcg[k]).mean())
    return res

## 4 · Train ALS **trong index-space của chainRec**

Ma trận positive = các **recommend-edge** (rating≥4) trong `data_train`, dùng đúng `user_idx`/`item_idx`
của chainRec → `U (N_USER×F)`, `V (N_ITEM×F)` **đã ở đúng index** để `rank_eval` chấm chung pool 1.57M.
Hyperparams khớp Phase 2 (`factors=64, iters=20, reg=0.1, alpha=40`). Vanilla ALS = confidence đồng nhất
(giá trị 1.0); biến thể F3-weighted để dành cho S3.

In [ ]:
pos = data_train[data_train[:,2] == REC]            # recommend-edge = rating>=4
rows = pos[:,0].astype(np.int32); cols = pos[:,1].astype(np.int32)
vals = np.ones(len(pos), dtype=np.float32)
user_item = csr_matrix((vals, (rows, cols)), shape=(N_USER, N_ITEM))
print(f"ALS train matrix: {user_item.shape}  nnz={user_item.nnz:,}")

ALS_FACTORS = 64
try:
    als = implicit.als.AlternatingLeastSquares(factors=ALS_FACTORS, iterations=20,
            regularization=0.1, alpha=40, random_state=SEED, use_gpu=False)
except TypeError:   # implicit cũ không nhận 'alpha' ở constructor
    als = implicit.als.AlternatingLeastSquares(factors=ALS_FACTORS, iterations=20,
            regularization=0.1, random_state=SEED, use_gpu=False)
    user_item = user_item * 40.0
t0 = time.time(); als.fit(user_item); print(f"ALS fit done in {time.time()-t0:.1f}s")
U = np.asarray(als.user_factors, dtype=np.float32)
V = np.asarray(als.item_factors, dtype=np.float32)
print("U", U.shape, " V", V.shape)
np.save(S2_LOCAL/"als_user_factors.npy", U)
np.save(S2_LOCAL/"als_item_factors.npy", V)        # ~400MB (regen được, mặc định không push)

item_pop = np.bincount(cols, minlength=N_ITEM).astype(np.float32)   # itemPop baseline

## 5 · Head-to-head: chainRec vs ALS vs itemPop — cùng `data_test`, cùng pool

`rank_eval` lọc test theo `pos_stage=REC` + sample `EVAL_USERS` bằng **cùng seed** → mọi model
chấm trên **đúng tập user/positive/negative giống nhau** ⇒ so sánh fair tuyệt đối.

In [ ]:
scorers = {}
for s in ["uniform", "stagewise"]:                  # chainRec — ưu tiên ckpt R@10 của S1a
    m, _ = build_model(s)
    try:
        sd = torch.load(ckpt_path(s, "_r10"), map_location=DEVICE)
    except Exception:
        sd = torch.load(ckpt_path(s), map_location=DEVICE)
        print(f"  ({s}) không thấy *_r10 → dùng ckpt S0")
    m.load_state_dict(sd)
    scorers[f"chainRec({s})"] = make_chainrec_scorer(m, REC)
scorers["ALS(vanilla)"] = make_als_scorer(U, V, device=DEVICE)
scorers["itemPop"]      = make_pop_scorer(item_pop, device=DEVICE)

res = {}
for name, fn in scorers.items():
    r = rank_eval(fn, data_test, user_item_map, N_ITEM, pos_stage=REC,
                  K_list=K_LIST, batch_users=BATCH_USERS, n_eval_users=EVAL_USERS, device=DEVICE)
    res[name] = r
    print(f"[{name:18}] AUC={r['AUC']:.4f}  R@10={r['Recall@10']:.4f}  "
          f"N@10={r['NDCG@10']:.4f}  R@20={r['Recall@20']:.4f}  (n={r['n_eval']})")

print("\n=== HEAD-TO-HEAD full-ranking (cùng data_test / pool / mask / users) ===")
print(f"{'model':20}{'AUC':>9}{'R@10':>9}{'N@10':>9}{'R@20':>9}")
for name, r in res.items():
    print(f"{name:20}{r['AUC']:>9.4f}{r['Recall@10']:>9.4f}{r['NDCG@10']:>9.4f}{r['Recall@20']:>9.4f}")
json.dump(res, open(S2_LOCAL/"s2_headtohead.json", "w"), indent=2)
print("\nSaved s2_headtohead.json")

## 6 · (Cho S3) Cầu nối ID `int (UCSD) ↔ string (reviews)`

Chuỗi map: `chainRec idx → raw int id (item_idx/user_idx nghịch) → string id (UCSD *_id_map.csv) → reviews`.
Dùng map chính thức của UCSD. Cần **Internet On**. Sản phẩm `id_bridge.pkl` để S3 join review-length (F3)
vào đúng (user_idx, item_idx) của chains.

In [ ]:
import urllib.request, pandas as pd
BASE = "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/"
def dl(fn):
    dst = Path("/kaggle/working")/fn
    if not dst.exists():
        print("downloading", fn, "..."); urllib.request.urlretrieve(BASE+fn, dst)
    return dst

RUN_BRIDGE = True     # cần Internet On
if RUN_BRIDGE:
    item_idx = load_pkl("item_idx.pkl")    # raw_int_book_id -> chainRec idx
    user_idx = load_pkl("user_idx.pkl")    # raw_int_user_id -> chainRec idx
    idx2raw_book = {v: k for k, v in item_idx.items()}
    idx2raw_user = {v: k for k, v in user_idx.items()}

    bmap = pd.read_csv(dl("book_id_map.csv"))   # book_id_csv,book_id(str)
    umap = pd.read_csv(dl("user_id_map.csv"))   # user_id_csv,user_id(str)
    print("book_id_map cols:", list(bmap.columns), "| user_id_map cols:", list(umap.columns))
    b_csv2str = dict(zip(bmap["book_id_csv"], bmap["book_id"].astype(str)))
    u_csv2str = dict(zip(umap["user_id_csv"], umap["user_id"].astype(str)))

    idx2book_str = np.empty(N_ITEM, dtype=object)
    for idx, raw in idx2raw_book.items(): idx2book_str[idx] = b_csv2str.get(int(raw))
    idx2user_str = np.empty(N_USER, dtype=object)
    for idx, raw in idx2raw_user.items(): idx2user_str[idx] = u_csv2str.get(int(raw))

    pickle.dump({"idx2book_str": idx2book_str, "idx2user_str": idx2user_str},
                open(S2_LOCAL/"id_bridge.pkl", "wb"))
    nb = int(sum(x is not None for x in idx2book_str))
    nu = int(sum(x is not None for x in idx2user_str))
    print(f"bridge: books mapped {nb:,}/{N_ITEM:,} | users mapped {nu:,}/{N_USER:,}")
    print("sample  idx0 book ->", idx2book_str[0], " | idx0 user ->", idx2user_str[0])
else:
    print("RUN_BRIDGE=False — bỏ qua (bật khi cần dựng artifact cho S3).")

## 7 · Push artifact + bước tiếp theo (S3)

In [ ]:
to_push = sorted(S2_LOCAL.glob("*"))
print("HF_TOKEN set?", HF_TOKEN is not None)
print("Files:", [(p.name, f"{p.stat().st_size/1e6:.1f}MB") for p in to_push])
if HF_TOKEN and to_push:
    from huggingface_hub import HfApi
    api = HfApi()
    for p in to_push:
        if p.stat().st_size > 200e6:            # als_item_factors.npy ~400MB: regen được, bỏ qua
            print("  skip (quá lớn, regen bằng cách chạy lại mục 4):", p.name); continue
        api.upload_file(path_or_fileobj=str(p), path_in_repo=f"s2/{p.name}",
                        repo_id=HF_REPO, repo_type="dataset", token=HF_TOKEN)
        print("  ✓ pushed", p.name)
    print(f"\nDone → {HF_REPO}/s2/")
else:
    print("Không push (thiếu token hoặc không có file). Artifact ở /kaggle/working/s2.")

## 8 · Đọc kết quả & S3

**Đọc head-to-head (mục 5):**
- So **AUC**: `chainRec(stagewise)` vs `ALS(vanilla)` trên *cùng dữ liệu*. chainRec hơn ⇒ mô hình hoá
  chuỗi hành vi đơn điệu có giá trị; hoà/thua ⇒ trên Goodreads MF thuần đã đủ (phù hợp nhận định
  "Goodreads bất lợi cho chainRec" của paper). `itemPop` là sàn — mọi model phải vượt rõ.
- R@10/N@10 nhỏ (~0.05) là bình thường (full-ranking 1.57M); chỉ dùng so **nội bộ**, không so paper.

**Cảnh báo công bằng:** ALS ở đây là **vanilla** (confidence đồng nhất, không bias item) — đại diện MF
chuẩn. Đừng so trực tiếp với Recall@10=0.6122 của ALS Phase 2: số đó là *sampled@500 trên reviews*,
khác hoàn toàn protocol full-ranking ở đây.

**Tiếp theo — S3 (đóng góp chính: edge-weighted edgewise loss):**
1. Dùng `id_bridge.pkl` (mục 6) join `review_token_count` (từ reviews parquet) vào từng recommend-edge
   của `data_train` → vector `w_pos` (F3 = (rating/5)·log1p(token_count)).
2. Train chainRec với `edgewise_loss(..., w_pos=w_pos)` (hook đã có sẵn từ S0/S1a).
3. Eval bằng đúng `rank_eval` này; so **chainRec vanilla vs chainRec+F3 vs ALS vanilla vs ALS+F3** (≥3 seed).
   Kỳ vọng F3 > vanilla +1–3% (recommend stage) — kết quả phẳng/âm vẫn có giá trị khoa học.
